In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [2]:
# Load data
df_train = pd.read_csv("/kaggle/input/compressed-cic-2018/df_train.csv")
df_test = pd.read_csv("/kaggle/input/compressed-cic-2018/df_test.csv")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")
print(f"\nTrain label distribution:\n{df_train['label'].value_counts()}")
print(f"\nTest label distribution:\n{df_test['label'].value_counts()}")

Train shape: (361660, 8)
Test shape: (101001, 8)

Train label distribution:
label
0    334346
1     21860
4      4390
2      1008
3        56
Name: count, dtype: int64

Test label distribution:
label
8    53996
0    45283
9     1722
Name: count, dtype: int64


In [3]:
# Prepare features and labels
feature_cols = ['latent_0', 'latent_1', 'latent_2', 'latent_3', 'latent_4', 'recon_loss', 'kld_loss']

X_train_full = df_train[feature_cols].values
y_train_full = df_train['label'].values

X_test = df_test[feature_cols].values
y_test = df_test['label'].values

# Compute inverse class weights
sample_weights_full = compute_sample_weight(class_weight='balanced', y=y_train_full)

print(f"X_train_full: {X_train_full.shape}, X_test: {X_test.shape}")
print(f"\nClass distribution in training:")
unique, counts = np.unique(y_train_full, return_counts=True)
for c, cnt in zip(unique, counts):
    print(f"  Class {c}: {cnt} samples, weight: {len(y_train_full) / (len(unique) * cnt):.4f}")

X_train_full: (361660, 7), X_test: (101001, 7)

Class distribution in training:
  Class 0: 334346 samples, weight: 0.2163
  Class 1: 21860 samples, weight: 3.3089
  Class 2: 1008 samples, weight: 71.7579
  Class 3: 56 samples, weight: 1291.6429
  Class 4: 4390 samples, weight: 16.4765


# Optuna Hyperparameter Tuning

In [4]:
def objective(trial):
    params = {
        'objective': 'multi:softmax',
        'num_class': len(np.unique(y_train_full)),
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'mlogloss',
        'random_state': 42,
        
        # Widened hyperparameter search space
        'n_estimators': trial.suggest_int('n_estimators', 50, 2000),
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.5, log=True),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.4, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-10, 100.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-10, 100.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-10, 10.0, log=True),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5, 5.0),
    }
    
    # Stratified K-Fold cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []
    
    for train_idx, val_idx in skf.split(X_train_full, y_train_full):
        X_train_fold = X_train_full[train_idx]
        y_train_fold = y_train_full[train_idx]
        X_val_fold = X_train_full[val_idx]
        y_val_fold = y_train_full[val_idx]
        sw_train_fold = sample_weights_full[train_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_train_fold, y_train_fold,
            sample_weight=sw_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            verbose=False
        )
        
        y_pred = model.predict(X_val_fold)
        f1 = f1_score(y_val_fold, y_pred, average='weighted')
        f1_scores.append(f1)
    
    return np.mean(f1_scores)

# Run Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"\nBest trial F1: {study.best_trial.value:.4f}")
print(f"Best params: {study.best_trial.params}")

[I 2026-01-03 14:05:30,911] A new study created in memory with name: no-name-90886d7e-be0d-4d0f-82e9-cb45c401deeb


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-01-03 14:07:13,441] Trial 0 finished with value: 0.9365805027869522 and parameters: {'n_estimators': 1733, 'max_depth': 9, 'learning_rate': 0.003473963594711077, 'subsample': 0.446197817321169, 'colsample_bytree': 0.6170990069598956, 'colsample_bylevel': 0.9950893915288075, 'colsample_bynode': 0.6918956040669391, 'min_child_weight': 5, 'reg_alpha': 1.4908111819962303e-05, 'reg_lambda': 1.745693630558042e-05, 'gamma': 3.6700246634401055e-10, 'max_delta_step': 7, 'scale_pos_weight': 2.5141189581405445}. Best is trial 0 with value: 0.9365805027869522.
[I 2026-01-03 14:07:57,551] Trial 1 finished with value: 0.918985237707331 and parameters: {'n_estimators': 1270, 'max_depth': 20, 'learning_rate': 0.0836765119306728, 'subsample': 0.8925231200274201, 'colsample_bytree': 0.6206745402903511, 'colsample_bylevel': 0.7323393259523091, 'colsample_bynode': 0.640638656345211, 'min_child_weight': 6, 'reg_alpha': 1.1106492355107753e-10, 'reg_lambda': 5.853071814355555e-06, 'gamma': 0.02196967

# Train Final Model with Best Params

In [5]:
# Train final model on full training data with best params
best_params = study.best_trial.params
best_params.update({
    'objective': 'multi:softmax',
    'num_class': len(np.unique(y_train_full)),
    'tree_method': 'hist',
    'device': 'cuda',
    'eval_metric': 'mlogloss',
    'random_state': 42,
})

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train_full, y_train_full, sample_weight=sample_weights_full, verbose=True)

print("Final model trained on full training data with class weights!")

Final model trained on full training data with class weights!


# Final Evaluation on Test Set

In [6]:
# Evaluate on test set
y_pred_test = final_model.predict(X_test)

# Compute sample weights for test set (multiclass)
sample_weights_test = compute_sample_weight(class_weight='balanced', y=y_test)

# Create binary labels for test: 0 = benign, 1 = attack (any non-0)
y_test_binary = (y_test != 0).astype(int)
y_pred_binary = (y_pred_test != 0).astype(int)

# Compute binary class weights (inverse frequency for 0 vs 1)
sample_weights_binary = compute_sample_weight(class_weight='balanced', y=y_test_binary)

# Custom weighted binary accuracy
def weighted_binary_accuracy(y_true_bin, y_pred_bin, sample_weights):
    correct_mask = y_true_bin == y_pred_bin
    
    benign_mask = y_true_bin == 0
    attack_mask = y_true_bin == 1
    
    benign_correct_weighted = np.sum(sample_weights[benign_mask & correct_mask])
    benign_total_weighted = np.sum(sample_weights[benign_mask])
    
    attack_correct_weighted = np.sum(sample_weights[attack_mask & correct_mask])
    attack_total_weighted = np.sum(sample_weights[attack_mask])
    
    total_correct_weighted = benign_correct_weighted + attack_correct_weighted
    total_weighted = np.sum(sample_weights)
    
    print(f"\nWeighted Binary Evaluation (Attack vs Benign):")
    print(f"  Benign (0) weighted acc: {benign_correct_weighted:.2f}/{benign_total_weighted:.2f} = {benign_correct_weighted/benign_total_weighted:.4f}")
    print(f"  Attack (1) weighted acc: {attack_correct_weighted:.2f}/{attack_total_weighted:.2f} = {attack_correct_weighted/attack_total_weighted:.4f}")
    
    return total_correct_weighted / total_weighted

print("=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)

# Binary class distribution
n_benign = np.sum(y_test_binary == 0)
n_attack = np.sum(y_test_binary == 1)
print(f"\nBinary test distribution: Benign={n_benign}, Attack={n_attack}")
print(f"Binary weights: Benign={len(y_test_binary)/(2*n_benign):.4f}, Attack={len(y_test_binary)/(2*n_attack):.4f}")

print(f"\nAccuracy (multiclass):  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Accuracy (multiclass weighted): {accuracy_score(y_test, y_pred_test, sample_weight=sample_weights_test):.4f}")
print(f"Accuracy (binary): {accuracy_score(y_test_binary, y_pred_binary):.4f}")
weighted_bin_acc = weighted_binary_accuracy(y_test_binary, y_pred_binary, sample_weights_binary)
print(f"Accuracy (binary weighted): {weighted_bin_acc:.4f}")

print(f"\nF1 (weighted): {f1_score(y_test, y_pred_test, average='weighted'):.4f}")
print(f"Precision (weighted): {precision_score(y_test, y_pred_test, average='weighted'):.4f}")
print(f"Recall (weighted): {recall_score(y_test, y_pred_test, average='weighted'):.4f}")

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Multiclass)")
print("=" * 60)
print(classification_report(y_test, y_pred_test))

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Binary: Attack vs Benign)")
print("=" * 60)
print(classification_report(y_test_binary, y_pred_binary, target_names=['Benign', 'Attack']))

FINAL TEST SET EVALUATION

Binary test distribution: Benign=45283, Attack=55718
Binary weights: Benign=1.1152, Attack=0.9064

Accuracy (multiclass):  0.4474
Accuracy (multiclass weighted): 0.3326
Accuracy (binary): 0.9764

Weighted Binary Evaluation (Attack vs Benign):
  Benign (0) weighted acc: 50390.09/50500.50 = 0.9978
  Attack (1) weighted acc: 48428.56/50500.50 = 0.9590
Accuracy (binary weighted): 0.9784

F1 (weighted): 0.4368
Precision (weighted): 0.4268
Recall (weighted): 0.4474

CLASSIFICATION REPORT (Multiclass)
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     45283
           1       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         0
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           8       0.00      0.00      0.00     53996
           9       0.00      0.00      0.00      1722

    accuracy                           

In [7]:
# Save the model
final_model.save_model("/kaggle/working/xgboost_model.json")
print("Model saved to /kaggle/working/xgboost_model.json")

Model saved to /kaggle/working/xgboost_model.json
